# OpenFoodFacts Data Cleaning

Cleaning pipeline for the following columns:

- `categories`, `categories_en`, `categories_tags`
- `nutriscore_score`, `nutriscore_grade`
- `allergens`
- `states_en`
- `first_packaging_code_geo`
- `data_quality_errors_tags`
- `emb_codes`, `emb_codes_tags`

## 1. Imports and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv(
    "openfoodfacts.csv",
    sep="\t",
    low_memory=False,
    on_bad_lines="skip"
)

## 2. Drop Redundant and Invalid Columns

- `allergens_en`: only 1 non-null entry, which is also invalid
- `states` and `states_tags`: identical content to `states_en`; `states_en` is kept as the sole states column

In [ ]:
df = df.drop(columns=["allergens_en"])
df = df.drop(columns=["states", "states_tags"])

## 3. Shared Constants

`INVALID` collects common placeholder strings that represent missing data across multiple columns.

In [ ]:
INVALID = {
    "", " ", "?", ".", ",", "n-a", "na", "none", "null", "0",
    "en:null", "en:none"
}

## 4. Categories Cleaning

**`categories`**: strip whitespace, lowercase, remove INVALID placeholders, non-food entries, undefined variants, pure-numeric values, and entries of 2 characters or fewer.

**`categories_en`**: same as above, plus translate known foreign-language entries to English and nullify remaining unresolved foreign-prefixed single tags.

**`categories_tags`**: dropped — carries identical information to `categories_en` in the 80% of data that matters, but in a less readable format (`en:` prefixes and hyphens instead of spaces).

In [ ]:
NON_FOOD_CATEGORIES = {
    'beauty', 'make-up', 'verzorging', 'parfum',
    'indoor and outdoor paints and varnishes',
    'tissue paper and tissue products',
    'hard surface cleaning products',
    'wood- cork- and bamboo-based floor coverings',
    'open beauty facts', 'non food products',
    'nl:beauty', 'nl:nagellakremover', 'nl:nagels', 'nl:nagelverzorging',
    'fr:indoor-and-outdoor-paints-and-varnishes',
    'fr:hard-covering-products',
    'en:non-food-products',
}

# Known foreign-language entries in categories_en mapped to English equivalents
CATEGORIES_EN_TRANSLATION = {
    'fr:jus': 'fruit juices',
    'fr:pâtes à tartiner': 'spreads',
    'fr:charcuteries cuites': 'cooked meats',
    'fr:charcuteries diverses': 'cured meats',
    'fr:seves-de-bouleau': np.nan,
    'fr:viander': np.nan,
    'fr:farine-maya': np.nan,
    'nl:beauty': np.nan,
    'nl:nagellakremover': np.nan,
    'nl:nagels': np.nan,
    'nl:nagelverzorging': np.nan,
    'hu:extrudált-kukorica': np.nan,
    'ru:котлета': np.nan,
    'ru:полуфабрикаты': np.nan,
    'fr:indoor-and-outdoor-paints-and-varnishes': np.nan,
    'fr:hard-covering-products': np.nan,
}

# Matches unresolved foreign-prefixed single tags e.g. 'fr:something'
FOREIGN_PREFIX_PATTERN = re.compile(r'^[a-z]{2}:[a-zA-Z]', re.IGNORECASE)

In [ ]:
def clean_categories(value):
    if not isinstance(value, str):
        return np.nan
    value = value.strip().lower()
    if value in INVALID:
        return np.nan
    if value in {'undefined', 'en:undefined', 'en:none'}:
        return np.nan
    if value in NON_FOOD_CATEGORIES:
        return np.nan
    if value.replace(',', '').replace(' ', '').isnumeric():
        return np.nan
    if len(value) <= 2:
        return np.nan
    return value


def clean_categories_en(value):
    if not isinstance(value, str):
        return np.nan
    value = value.strip()
    value_lower = value.lower()
    if value_lower in INVALID:
        return np.nan
    if value_lower in {'undefined', 'null', 'none'}:
        return np.nan
    if value_lower in NON_FOOD_CATEGORIES:
        return np.nan
    if value_lower.replace(',', '').replace(' ', '').isnumeric():
        return np.nan
    if len(value) <= 2:
        return np.nan
    if value_lower in CATEGORIES_EN_TRANSLATION:
        return CATEGORIES_EN_TRANSLATION[value_lower]
    # Nullify unresolved single foreign-prefixed tags
    # (comma-separated lists that contain one foreign tag are kept)
    if ',' not in value and FOREIGN_PREFIX_PATTERN.match(value):
        return np.nan
    return value


df['categories'] = df['categories'].apply(clean_categories)
df['categories_en'] = df['categories_en'].apply(clean_categories_en)
df = df.drop(columns=['categories_tags'])

## 5. Nutriscore Score Cleaning

`pd.to_numeric` with `errors='coerce'` converts any non-numeric entry to NaN. Note: `0` is a valid score (the boundary between grade B and C) and is intentionally preserved — the INVALID set is not applied here.

In [ ]:
df["nutriscore_score"] = pd.to_numeric(df["nutriscore_score"], errors="coerce")

## 6. Nutriscore Grade Cleaning

1. Lowercase and strip whitespace
2. Replace `'unknown'` (missing information) and `'590'` (data entry error) with NaN
3. Fill remaining NaN and `'not-applicable'` entries from `labels_en` using a regex that extracts grade letters embedded in label strings such as `'Nutri-Score Grade A'`

In [ ]:
df["nutriscore_grade"] = (
    df["nutriscore_grade"]
    .str.lower()
    .str.strip()
)

df["nutriscore_grade"] = df["nutriscore_grade"].replace({
    "unknown": np.nan,
    "590": np.nan
})

In [ ]:
def extract_nutriscore_grade(label):
    """
    Extracts a Nutri-Score grade letter (a-e) from a labels_en string.
    Matches 'nutriscore' or 'nutri-score', skips any intervening text,
    then captures the first grade letter a through e.
    """
    if not isinstance(label, str):
        return None
    match = re.search(r'nutri.?score[^a-e]*([a-e])', label.lower())
    if match:
        return match.group(1)
    return None


extracted = df['labels_en'].apply(extract_nutriscore_grade)

# Only fill rows that are genuinely missing or flagged as not-applicable
# — never overwrite an existing valid grade
fill_mask = df['nutriscore_grade'].isna() | (df['nutriscore_grade'] == 'not-applicable')
valid_extracted = extracted.notna()

df.loc[fill_mask & valid_extracted, 'nutriscore_grade'] = (
    extracted[fill_mask & valid_extracted]
)

## 7. Allergens Cleaning

- Strip whitespace and lowercase
- Remove common INVALID placeholders
- Remove purely numeric entries (e.g. `'6'`)
- **`'none'` and `'en:none'` are kept** — they are valid entries indicating a product contains no allergens

In [ ]:
def clean_allergens(value):
    if not isinstance(value, str):
        return np.nan
    value = value.strip().lower()
    # Custom invalid set — excludes 'none' and 'en:none' which are valid
    allergen_invalid = {
        "", " ", "?", ".", ",", "n-a", "na", "null", "0", "en:null"
    }
    if value in allergen_invalid:
        return np.nan
    if value.replace(',', '').replace(' ', '').isnumeric():
        return np.nan
    return value


df["allergens"] = df["allergens"].apply(clean_allergens)

## 8. States Cleaning

`states` and `states_tags` were dropped in section 2. Only `states_en` is kept.

- Strip whitespace
- Lowercase — safe here because states form a controlled vocabulary; inconsistent casing would cause the same state to be counted as two distinct categories in any groupby or value_counts operation

In [ ]:
def clean_states(value):
    if not isinstance(value, str):
        return np.nan
    return value.strip().lower()


df["states_en"] = df["states_en"].apply(clean_states)

## 9. First Packaging Code Geo Cleaning

`first_packaging_code_geo` stores coordinates as `'lat,lon'` strings. Each value is validated against:

- Must split into exactly two parts on the comma
- Both parts must parse as floats
- Latitude must be within −90 to 90
- Longitude must be within −180 to 180

Anything failing validation (including a misplaced food category label found during investigation) is replaced with NaN.

In [ ]:
def parse_geo(val):
    try:
        parts = str(val).split(',')
        if len(parts) == 2:
            lat, lon = float(parts[0]), float(parts[1])
            if -90 <= lat <= 90 and -180 <= lon <= 180:
                return val
    except:
        pass
    return np.nan


df['first_packaging_code_geo'] = df['first_packaging_code_geo'].apply(parse_geo)

## 10. Data Quality Errors Tags Cleaning

- Strip whitespace and lowercase
- Remove INVALID set placeholders

Investigation confirmed all top-80% values follow the valid `en:<error-type>` format — no further cleaning is needed beyond standardisation.

In [ ]:
def clean_data_quality_errors_tags(value):
    if not isinstance(value, str):
        return np.nan
    value = value.strip().lower()
    if value in INVALID:
        return np.nan
    return value


df['data_quality_errors_tags'] = df['data_quality_errors_tags'].apply(
    clean_data_quality_errors_tags
)

## 11. EMB Codes Cleaning

EMB codes identify the EU facility where a food product was packaged. Valid codes follow patterns such as `EMB 12345A`, `FR 72.264.002 EC`, or `FSC-C014047`.

Issues found and addressed:

- Common INVALID placeholders
- Language/country code fragments (`fr`, `de`, `ec`, `ce`, etc.)
- Placeholder words (`fabricante`, `envasador`, `sans`, `inconnu`, etc.)
- Short fragments of 3 characters or fewer
- Purely numeric strings (barcodes or lot numbers entered in the wrong field)
- The Julian calendar sentence slug
- Slugified company names detected via common legal suffixes (`-s-a`, `-gmbh`, `-ltd`, etc.)
- Random short alphanumeric fragments like `a879`, `b01h`, `1687` (pattern: optional letter + 2–4 digits + optional letter)

In [ ]:
EMB_INVALID_VALUES = {
    # Language/country code fragments
    'fr', 'de', 'es', 'it', 'nl', 'pl', 'uk', 'ue', 'eg', 'ec', 'ce',
    'eu', 'be', 'at', 'ch', 'pt', 'se', 'dk', 'fi', 'hu', 'cz', 'sk',
    'ro', 'bg', 'hr', 'si', 'lt', 'lv', 'ee', 'gr', 'ie', 'no', 'y',
    # Placeholder words — raw emb_codes format
    'sans', 'absent', 'non', 'neant', 'aucun', 'inconnu', 'maroc',
    'deutschland', 'fao 87', 'no indica', 'not indicated', 'pieces',
    'fabricante', 'envasador', 'otros', 'desconocido', 'comercializador',
    'distribuidor', 'importador',
    # Placeholder words — normalised emb_codes_tags format
    'sans-estampille', 'not-indicated', 'non-indique', 'non-precise',
    'inexistant', 'neant', 'aucun', 'inconnu',
    'fabricante-y-envasador', 'distribuidor-en-espana',
    'neprecizat', 'perteneciente-a', 'fc-01', 'sif',
}

# Long Julian calendar sentence normalised into a hyphenated slug
LONG_SLUG_PATTERN = re.compile(r'l-code-l-first-digit', re.IGNORECASE)

# Slugified company names end with common legal suffixes
COMPANY_SUFFIX_PATTERN = re.compile(
    r'(-s-a|-s-l|-s-a-s|-s-a-u|-s-l-u|-s-coop|-s-c-a|-gmbh|-ltd|'
    r'-plc|-n-v|-b-v|-s-p-a|-s-r-l|-s-c|-ag|-kg|-inc|-corp|-co|-group'
    r'|-holding|-holdings|-iberica|-espana|-espagne|-france|-italia'
    r'|-deutschland|-manufacturing|-alimentacion|-alimentaria|-alimentarios'
    r'|-productos|-conservas|-aceitunas|-citricos|-agricola|-agroalimentaria'
    r'|-commerciale|-international|-iberia)$',
    re.IGNORECASE
)

In [ ]:
def clean_emb_extended(value):
    if not isinstance(value, str):
        return np.nan
    value = value.strip().lower()
    if value in INVALID:
        return np.nan
    if value in EMB_INVALID_VALUES:
        return np.nan
    # Short fragments of 3 characters or fewer
    if len(value) <= 3:
        return np.nan
    # Purely numeric strings — also handles comma-separated numeric combos
    if value.replace(',', '').replace(' ', '').replace('-', '').isnumeric():
        return np.nan
    # Julian calendar sentence slug
    if LONG_SLUG_PATTERN.search(value):
        return np.nan
    # Slugified company names
    if COMPANY_SUFFIX_PATTERN.search(value):
        return np.nan
    # Random short alphanumeric fragments:
    # optional letter + 2–4 digits + optional letter
    # catches 'a879', 'b01h', '1687' but not real codes which contain hyphens
    if re.fullmatch(r'[a-z]?\d{2,4}[a-z]?', value):
        return np.nan
    return value


df['emb_codes'] = df['emb_codes'].apply(clean_emb_extended)
df['emb_codes_tags'] = df['emb_codes_tags'].apply(clean_emb_extended)

## 12. Remove `en:` Prefixes

The `en:` prefix is a language tag added by Open Food Facts to indicate an English-language entry. It carries no analytical meaning and reduces readability. It is stripped from `allergens`, `states_en`, and `data_quality_errors_tags` — the three columns where it appears as a prefix on individual tags.

Each column may contain comma-separated tags, so the prefix is removed from each tag individually. Tags that do not start with `en:` (e.g. foreign-language allergen entries) are left untouched.

**Note:** `en:none` in allergens becomes `none` after this step, which remains consistent with its meaning of 'no allergens present'.

In [ ]:
def remove_en_prefix(value):
    if not isinstance(value, str):
        return value
    tags = [tag.strip() for tag in value.split(',')]
    tags = [tag[3:] if tag.startswith('en:') else tag for tag in tags]
    return ','.join(tags)


for col in ['allergens', 'states_en', 'data_quality_errors_tags']:
    df[col] = df[col].apply(remove_en_prefix)